# 14 · Simulateur de scénarios 2022

**Projet 4 : Analyse des données de vente et retours** · Sprint 5 · Unité d'apprentissage 3

**Question métier** : *quel résultat d'exploitation l'entreprise peut-elle attendre en 2022 selon les décisions qu'elle prend, et quelles décisions restent les bonnes même si la prévision des ventes se trompe ?*

**Objectif de l'énoncé servi par ce notebook** : formuler des recommandations chiffrées pour améliorer la rentabilité.

**Le principe**

| Élément | Ce que c'est | D'où il vient |
|---|---|---|
| La base | Le chiffre d'affaires 2022 de chaque boutique si rien ne change | Lissage saisonnier par canal, avec deux variantes (2021 reconduit, Random Forest du notebook 10) |
| Le compte de résultat | Marge, frais généraux, publicité, résultat d'exploitation, boutique par boutique | Taux de marge et charges de 2021 (05b) |
| Les leviers | Les décisions de gestion, sous forme de réglages | Les mécanismes chiffrés aux notebooks 06, 07, 08 et 09, sans en inventer de nouveau |
| La robustesse | Les mêmes calculs avec une base fausse de ±10 %, une autre base, des hypothèses défavorables | Ce notebook |

Le moteur est dans `src/simulateur.py` : le même code servira à l'application Streamlit.

| Étape | Ce qu'on fait | Question posée |
|---|---|---|
| 2. Vérification | Le moteur refait l'exercice 2021 et chaque levier du 06 | Le simulateur calcule-t-il comme l'analyse ? |
| 3. Fil de l'eau | Base 2022, charges reconduites, résultat | Que coûte l'inaction ? |
| 4. Leviers | Chaque décision seule, en 2022 | Que rapporte chaque levier ? |
| 5. Scénarios | Quatre combinaisons de décisions | Que rapporte une politique cohérente ? |
| 6. Robustesse | Autres bases, base ±10 %, hypothèses incertaines | La recommandation tient-elle si la prévision se trompe ? |
| 7. Zones de décision | Coût d'une nouveauté, réaction des ventes à la publicité | Jusqu'où une décision reste-t-elle gagnante ? |

## 0. Préparation

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.holtwinters import ExponentialSmoothing

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# 1. Retrouver la racine du projet (le dossier qui contient src/ et data/)
# ---------------------------------------------------------------------------
racine = Path.cwd()
while not ((racine / "src").is_dir() and (racine / "data").is_dir()):
    if racine == racine.parent:
        raise FileNotFoundError("Racine du projet introuvable : lancer ce notebook depuis le dossier Projet ADD ou un de ses sous-dossiers.")
    racine = racine.parent
sys.path.insert(0, str(racine))

# ---------------------------------------------------------------------------
# 2. Modules du projet
# ---------------------------------------------------------------------------
#   config     -> chemins, années de l'étude
#   viz        -> charte graphique
#   analyse    -> libellés français, export des tableaux
#   simulateur -> moteur du simulateur (réutilisé par l'application Streamlit)
from src import config, viz, analyse, simulateur
from src.analyse import ORDRE_TYPES, NOMS_MOIS
from src.simulateur import reglages

config.creer_dossiers()
viz.appliquer_style()
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)

EXERCICE = config.EXERCICE                    # 2021 : le dernier exercice, point de départ
ANNEE = EXERCICE + 1                          # 2022 : l'année simulée
COULEURS_TYPES = {"En ligne": viz.CATEGORIES[2], "Revendeur": viz.ACCENT, "Magasins": viz.ALERTE}


def m_dollars(valeur, decimales=1, signe=False):
    """Montant en millions de dollars, virgule française : 12,3 M$."""
    texte = f"{valeur / 1e6:{'+' if signe else ''},.{decimales}f}".replace(",", " ").replace(".", ",")
    return f"{texte} M$"


def pct(valeur, decimales=1, signe=False):
    """Taux en pourcentage, virgule française : 0.07 -> 7,0 %."""
    return f"{valeur:{'+' if signe else ''}.{decimales}%}".replace(".", ",").replace("%", " %")


tableaux = {}                                 # tableaux exportés vers Excel à la fin
print("Racine du projet :", config.ROOT, f"· année simulée : {ANNEE}, à partir de l'exercice {EXERCICE}")

## 1. Chargement

In [ ]:
chemins = {
    "commun": config.AGREGATS / "ventes_charges_boutique_mois.parquet",
    "charges": config.CLEAN / "fait_charges.parquet",
    "ventes": config.CLEAN / "fait_ventes_variables.parquet",
    "dim_produit": config.CLEAN_DIMENSIONS / "dim_produit.parquet",
    "dim_boutique": config.CLEAN_DIMENSIONS / "dim_boutique.parquet",
}
for nom, chemin in chemins.items():
    if not chemin.exists():
        raise FileNotFoundError(f"{chemin} introuvable : exécuter d'abord les notebooks 00 à 05c.")

commun = pd.read_parquet(chemins["commun"])
charges = pd.read_parquet(chemins["charges"])
ventes = pd.read_parquet(chemins["ventes"], columns=["annee", "id_boutique", "id_produit", "type_boutique", "ca_net",
                                                     "taux_remise_annonce", "montant_remise", "marge_perdue_retours"])
dim_produit = pd.read_parquet(chemins["dim_produit"])
dim_boutique = pd.read_parquet(chemins["dim_boutique"])

donnees = simulateur.preparer(commun, charges, ventes, dim_produit, dim_boutique, EXERCICE)
boutiques = donnees["boutiques"]
print(f"{len(boutiques)} boutiques · exercice {EXERCICE} : CA net {m_dollars(boutiques.ca_net.sum())}, "
      f"résultat d'exploitation {m_dollars(boutiques.resultat_exploitation.sum())}")
print(f"Nouveautés de {EXERCICE} : {donnees['lances']} produits, {m_dollars(donnees['ca_par_nouveaute'], 3)} de CA chacune en moyenne")

## 2. Le moteur refait-il les calculs de l'analyse ?

Avant de simuler 2022, le moteur est appliqué à l'exercice 2021, sans aucune hausse de charges. Il doit retrouver le résultat réel, puis le gain de chaque levier tel que l'ont chiffré les notebooks 06 et 09.

In [ ]:
base_2021 = simulateur.base_reconduite(donnees)
depart = simulateur.simuler(donnees, base_2021, reglages(hausse_charges="gelée"))
resultat_depart = simulateur.resultat_entreprise(depart)
print(f"Résultat d'exploitation {EXERCICE} recalculé par le moteur : {m_dollars(resultat_depart, 2)} "
      f"(réel : {m_dollars(boutiques.resultat_exploitation.sum(), 2)})")

# Chaque levier, seul, appliqué à l'exercice 2021 : le moteur doit retrouver les chiffres du notebook 06
CONTROLES = [  # (levier, réglages, gain publié au 06 ou au 09, en M$)
    ("Revenir au rythme de lancement de 2019 (552 nouveautés)", dict(lancements=552), 46.1),
    ("Vendre les nouveautés en magasin comme les produits anciens", dict(nouveautes_en_magasin=1.0), 3.8),
    ("Ramener personnel, administration et informatique au niveau de 2019", dict(frais_maitrisables="niveau 2019"), 9.2),
    ("Ramener les mêmes comptes au niveau de 2020", dict(frais_maitrisables="niveau 2020"), 4.8),
    ("Regrouper administration et informatique par pays (30 %)", dict(regroupement_pays=0.30), 6.6),
    ("Baisser la publicité des magasins de 10 %", dict(baisse_publicite_magasins=0.10), 6.2),
    ("Baisser la publicité des magasins de 20 %", dict(baisse_publicite_magasins=0.20), 12.4),
    ("Supprimer les remises en magasin", dict(remises_magasin="supprimées"), 5.4),
    ("Suspendre la promotion à 20 % en magasin", dict(remises_magasin="sans le 20 %"), 3.9),
    ("Plafonner la remise à 10 % en ligne et chez le revendeur", dict(plafond_remise_10_hors_magasin=True), 2.9),
    ("Réduire les retours de moitié", dict(baisse_retours=0.5), 2.6),
]
lignes = []
for nom, kw, publie in CONTROLES:
    r = simulateur.simuler(donnees, base_2021, reglages(hausse_charges="gelée", **kw))
    gain = simulateur.resultat_entreprise(r) - resultat_depart
    lignes.append({"Levier": nom, "Moteur (M$)": gain / 1e6, "Notebook 06 ou 09 (M$)": publie, "Écart (M$)": gain / 1e6 - publie})
verification = pd.DataFrame(lignes).set_index("Levier")
display(verification.style.format("{:+.2f}", subset=["Écart (M$)"]).format("{:.2f}", subset=["Moteur (M$)", "Notebook 06 ou 09 (M$)"]))
tableaux["2_verification_moteur"] = verification.reset_index()

**Lecture.** Le moteur retrouve le résultat d'exploitation 2021 au dollar près (141,65 M$), puis chacun des onze leviers à 0,04 M$ près ; les écarts viennent seulement des arrondis publiés. Le simulateur ne réinvente donc rien : il réutilise les mécanismes déjà démontrés et les applique à une autre année.

## 3. L'année 2022 au fil de l'eau

### 3.1 Le chiffre d'affaires de base

Le notebook 10 a montré qu'aucune méthode ne prévoit une année à mieux qu'environ ±10 % avec trois ans d'historique. La base retenue est le **lissage saisonnier par canal** : il part du niveau récent de chaque canal et applique sa saison, sans prolonger de tendance. Chaque boutique reçoit ensuite sa part du canal en 2021. Deux variantes servent au test de robustesse : 2021 reconduit, et la prévision du Random Forest du notebook 10.

In [ ]:
# ---------------------------------------------------------------------------
# 1. Chiffre d'affaires mensuel par canal, 2019 à 2021, et lissage saisonnier pour 2022
# ---------------------------------------------------------------------------
commun["type"] = commun.type_boutique.map(simulateur.LIBELLES_TYPES)
series = commun.pivot_table(index="date", columns="type", values="ca_net", aggfunc="sum")[ORDRE_TYPES]
series.index = pd.DatetimeIndex(series.index, freq="MS")
mois_futurs = pd.date_range(f"{ANNEE}-01-01", periods=12, freq="MS")
lissage = pd.DataFrame({t: ExponentialSmoothing(series[t], trend=None, seasonal="mul", seasonal_periods=12).fit().forecast(12).values
                        for t in ORDRE_TYPES}, index=mois_futurs)

bases_canal = pd.DataFrame({
    f"Réel {EXERCICE}": series[series.index.year == EXERCICE].sum(),
    f"Base {ANNEE} : lissage saisonnier": lissage.sum(),
})
chemin_rf = config.PREDICTIONS / "previsions_ventes.parquet"
if chemin_rf.exists():
    rf = pd.read_parquet(chemin_rf)
    rf = rf[rf.jeu.str.startswith("Prévision")]
    bases_canal[f"Variante : Random Forest (notebook 10)"] = rf.groupby("type").prevision.sum()
else:
    print("Prévisions du notebook 10 absentes : la variante Random Forest n'est pas calculée.")
bases_canal.loc["Total"] = bases_canal.sum()
evolution = bases_canal.iloc[:, 1:].div(bases_canal.iloc[:, 0], axis=0) - 1
display(bases_canal.map(m_dollars))
print(f"Évolution par rapport à {EXERCICE} :")
display(evolution.map(lambda v: pct(v, 1, signe=True)))
tableaux["3_bases_par_canal"] = bases_canal.reset_index(names="Canal")

BASES = {"Lissage saisonnier": simulateur.base_par_canal(donnees, lissage.sum()),
         f"{EXERCICE} reconduit": base_2021}
if chemin_rf.exists():
    BASES["Random Forest (notebook 10)"] = rf.groupby("id_boutique").prevision.sum().reindex(boutiques.id_boutique).fillna(
        base_2021) .rename("ca_base")
BASE = BASES["Lissage saisonnier"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, t in zip(axes, ORDRE_TYPES):
    ax.plot(series.index, series[t], color=viz.FONCE, lw=1.8, label="Réel")
    ax.plot(lissage.index, lissage[t], color=viz.ACCENT, lw=2, ls="--", label=f"Base {ANNEE}")
    ax.fill_between(lissage.index, lissage[t] * 0.9, lissage[t] * 1.1, color=viz.ACCENT, alpha=0.15, label="Fourchette ±10 %")
    viz.format_dollars(ax); ax.set_ylim(bottom=0); ax.set_title(t, fontsize=12)
    ax.xaxis.set_major_locator(mdates.YearLocator()); ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
axes[2].legend(loc="lower left")
fig.suptitle(f"Chiffre d'affaires net mensuel par canal, et base {ANNEE} au fil de l'eau", x=0.01, ha="left", fontweight="bold", color=viz.FONCE)
fig.tight_layout()
viz.enregistrer(fig, "14_base_par_canal")
plt.show()

**Lecture.**
- **La base prévoit −3,4 % pour l'entreprise** (664,8 M$), avec des trajectoires opposées : la boutique en ligne monte de 6,6 %, le revendeur recule de 1,5 %, les magasins perdent encore 14,8 %. Le lissage prolonge le niveau de fin 2021, qui était bas en magasin.
- **Le Random Forest prévoit presque la stabilité** (−0,5 %). L'écart entre les deux bases, 20 M$, est du même ordre que l'incertitude mesurée au notebook 10 : c'est précisément ce que la partie 6 doit tester.

### 3.2 Les charges et le résultat au fil de l'eau

« Au fil de l'eau » veut dire que la direction garde ses habitudes de budget : chaque compte de frais généraux augmente en 2022 comme il a augmenté en 2021. La publicité garde son budget de 2021. Les autres charges suivent les ventes.

In [ ]:
hausses = pd.DataFrame(donnees["hausses"]).T[ORDRE_TYPES]
hausses.index.name = "Compte"
print(f"Hausse de chaque compte entre {EXERCICE - 1} et {EXERCICE}, reconduite en {ANNEE} au fil de l'eau :")
display(hausses.map(lambda v: pct(v, 1, signe=True)))
tableaux["3_hausses_reconduites"] = hausses.reset_index()

fil = simulateur.simuler(donnees, BASE, reglages())
gel = simulateur.simuler(donnees, BASE, reglages(hausse_charges="gelée"))
synthese_fil = pd.DataFrame({
    f"Réel {EXERCICE}": depart.groupby("type").resultat_exploitation.sum(),
    f"{ANNEE}, charges gelées": gel.groupby("type").resultat_exploitation_simule.sum(),
    f"{ANNEE} au fil de l'eau": fil.groupby("type").resultat_exploitation_simule.sum()}).loc[ORDRE_TYPES]
synthese_fil.loc["Entreprise"] = synthese_fil.sum()
deficit = pd.Series({f"Réel {EXERCICE}": int((depart[depart.type == "Magasins"].resultat_exploitation < 0).sum()),
                     f"{ANNEE}, charges gelées": int((gel[gel.type == "Magasins"].resultat_exploitation_simule < 0).sum()),
                     f"{ANNEE} au fil de l'eau": int((fil[fil.type == "Magasins"].resultat_exploitation_simule < 0).sum())})
print("Résultat d'exploitation par canal :")
display(synthese_fil.map(m_dollars))
print("Magasins déficitaires (sur 54) :", " · ".join(f"{k} : {v}" for k, v in deficit.items()))
RESULTAT_FIL = simulateur.resultat_entreprise(fil)
tableaux["3_resultat_fil_de_l_eau"] = synthese_fil.reset_index(names="Canal")

**Lecture : le coût de l'inaction.**
- **Le résultat d'exploitation tombe de 141,7 M$ à 114,1 M$**, soit 27,6 M$ de moins en un an. La moitié vient de la baisse des ventes (141,7 → 128,7 M$ à charges gelées), l'autre moitié des hausses de charges reconduites (128,7 → 114,1 M$).
- **Les magasins passent de −10,8 M$ à −40,6 M$**, et 52 magasins sur 54 sont déficitaires. L'administration, à +15,9 % par an, reste le compte qui dérape le plus.
- **La boutique en ligne porte l'entreprise** : 106,0 M$ de résultat, plus qu'en 2021.

Ne rien décider n'est pas une option neutre : c'est une perte de résultat d'environ 28 M$ dès la première année.

## 4. Ce que rapporte chaque levier en 2022

Chaque levier est appliqué seul, et comparé au fil de l'eau sur la même base. Le gain est en résultat d'exploitation par an.

In [ ]:
LEVIERS = {  # levier -> (famille, réglages, hypothèse à garder en tête)
    "Revenir au rythme de lancement de 2019 (552 nouveautés)": ("Assortiment", dict(lancements=552), "gain brut : coût de lancement inconnu"),
    "Vendre les nouveautés en magasin comme les produits anciens": ("Assortiment", dict(nouveautes_en_magasin=1.0), "mise en rayon et en avant"),
    "Geler personnel, administration et informatique au niveau de 2021": ("Charges", dict(frais_maitrisables="gel"), "sans risque sur les ventes"),
    "Ramener ces comptes au niveau de 2019": ("Charges", dict(frais_maitrisables="niveau 2019"), "sans risque sur les ventes"),
    "Regrouper administration et informatique par pays (30 %)": ("Charges", dict(regroupement_pays=0.30), "piste de réflexion, à ne pas additionner au niveau 2019"),
    "Baisser la publicité des magasins de 10 %": ("Publicité", dict(baisse_publicite_magasins=0.10), "suppose des ventes inchangées"),
    "Supprimer les remises en magasin": ("Promotions", dict(remises_magasin="supprimées"), "les ventes remisées auraient eu lieu sans remise (09)"),
    "Plafonner la remise à 10 % en ligne et chez le revendeur": ("Promotions", dict(plafond_remise_10_hors_magasin=True), "à tester avec un groupe témoin"),
    "Réduire les retours de moitié": ("Retours", dict(baisse_retours=0.5), "aucun pattern de retour à cibler (08)"),
}


def gains_leviers(base, **hypotheses):
    """Gain de chaque levier, seul, par rapport au fil de l'eau sur la même base."""
    reference = simulateur.simuler(donnees, base, reglages(**hypotheses))
    ref = simulateur.resultat_entreprise(reference)
    lignes = []
    for nom, (famille, kw, hypothese) in LEVIERS.items():
        r = simulateur.simuler(donnees, base, reglages(**hypotheses, **kw))
        mag = r[r.type == "Magasins"].resultat_exploitation_simule
        lignes.append({"Levier": nom, "Famille": famille, "Gain par an": simulateur.resultat_entreprise(r) - ref,
                       "Résultat des magasins après": mag.sum(), "Magasins déficitaires après": int((mag < 0).sum()),
                       "Hypothèse": hypothese})
    return pd.DataFrame(lignes).set_index("Levier").sort_values("Gain par an", ascending=False)


leviers = gains_leviers(BASE)
mag_fil = fil[fil.type == "Magasins"].resultat_exploitation_simule
print(f"Au fil de l'eau {ANNEE} : résultat {m_dollars(RESULTAT_FIL)} · magasins {m_dollars(mag_fil.sum())}, "
      f"{int((mag_fil < 0).sum())} déficitaires sur 54")
display(leviers.style.format({"Gain par an": m_dollars, "Résultat des magasins après": m_dollars}))
tableaux["4_leviers_2022"] = leviers.reset_index()

COULEURS_FAMILLE = {"Assortiment": viz.PRINCIPALE, "Charges": viz.CATEGORIES[2], "Publicité": viz.ALERTE,
                    "Promotions": viz.ACCENT, "Retours": viz.GRIS}
fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(data=leviers.reset_index(), y="Levier", x="Gain par an", hue="Famille", dodge=False, palette=COULEURS_FAMILLE, ax=ax)
for y, v in enumerate(leviers["Gain par an"]):
    ax.text(v, y, "  " + m_dollars(v), va="center", fontsize=8.5, color=viz.FONCE)
viz.format_dollars(ax, axe="x"); ax.set_xlim(0, leviers["Gain par an"].max() * 1.2)
ax.grid(axis="x"); ax.grid(axis="y", visible=False); ax.set_ylabel(""); ax.set_xlabel(""); ax.legend(title="", loc="lower right")
ax.tick_params(axis="y", labelsize=8.5)
viz.titre(ax, f"Ce que rapporte chaque levier en {ANNEE}", "Gain de résultat d'exploitation par an, par rapport au fil de l'eau")
viz.enregistrer(fig, "14_leviers")
plt.show()

**Lecture.**
- **Renouveler l'assortiment reste de loin le premier levier** : 46,1 M$, comme en 2021, puisque le chiffre d'affaires d'une nouveauté ne dépend pas de la base. Il profite surtout à la boutique en ligne et au revendeur, qui vendent les deux tiers des nouveautés : seul, il laisse 50 magasins déficitaires.
- **Revenir aux charges de 2019 rapporte 15,1 M$ en 2022**, contre 9,2 M$ en 2021 : chaque année sans décision ajoute une nouvelle hausse à annuler. Simplement geler ces comptes au niveau de 2021 rapporte déjà 6,1 M$.
- **Aucun levier ne rend seul les magasins rentables** : le meilleur les laisse à −24,9 M$. Il faut combiner.
- Promotions, nouveautés en magasin et retours rapportent chacun 2,5 à 4,6 M$ : ce sont des compléments, pas des solutions.

## 5. Les scénarios

| Scénario | Décisions | Logique |
|---|---|---|
| Prudent | Geler personnel, administration et informatique ; suspendre la promotion à 20 % en magasin | Ce qu'on peut décider sans risque, dès maintenant |
| Charges | Revenir aux charges de 2019 ; publicité des magasins −10 % ; plus de remises en magasin | Traiter la conséquence (charges dimensionnées sur les ventes de 2019) |
| Assortiment | 552 nouveautés (rythme de 2019) ; nouveautés en magasin comme les produits anciens | Traiter la cause (assortiment qui vieillit) |
| Complet | Assortiment et charges ensemble | La recommandation de l'UA2 : la cause, puis la conséquence |

Le regroupement des bureaux par pays n'entre dans aucun scénario : il recoupe le retour aux charges de 2019 et reste une piste de réflexion pour le métier.

In [ ]:
SCENARIOS = {
    "Au fil de l'eau": {},
    "Prudent": dict(frais_maitrisables="gel", remises_magasin="sans le 20 %"),
    "Charges": dict(frais_maitrisables="niveau 2019", baisse_publicite_magasins=0.10, remises_magasin="supprimées"),
    "Assortiment": dict(lancements=552, nouveautes_en_magasin=1.0),
    "Complet": dict(lancements=552, nouveautes_en_magasin=1.0, frais_maitrisables="niveau 2019",
                    baisse_publicite_magasins=0.10, remises_magasin="supprimées"),
}


def jouer(base, scenarios=SCENARIOS, **hypotheses):
    """Résultat de chaque scénario sur une base donnée."""
    lignes, details = [], {}
    for nom, kw in scenarios.items():
        r = simulateur.simuler(donnees, base, reglages(**hypotheses, **kw))
        mag = r[r.type == "Magasins"].resultat_exploitation_simule
        details[nom] = r
        lignes.append({"Scénario": nom, "CA net": r.ca_net_simule.sum(), "Résultat d'exploitation": simulateur.resultat_entreprise(r),
                       "Résultat des magasins": mag.sum(), "Magasins déficitaires": int((mag < 0).sum())})
    t = pd.DataFrame(lignes).set_index("Scénario")
    t["Gain par rapport au fil de l'eau"] = t["Résultat d'exploitation"] - t.loc["Au fil de l'eau", "Résultat d'exploitation"]
    return t, details


scenarios, details = jouer(BASE)
description = pd.Series({nom: ", ".join(f"{k} = {v}" for k, v in kw.items()) or "aucune décision" for nom, kw in SCENARIOS.items()},
                        name="Réglages")
display(scenarios.style.format({c: m_dollars for c in ["CA net", "Résultat d'exploitation", "Résultat des magasins", "Gain par rapport au fil de l'eau"]}))
display(description.to_frame())
tableaux["5_scenarios"] = scenarios.join(description).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 4.3), gridspec_kw={"width_ratios": [1.4, 1]})
t = scenarios.reset_index()
sns.barplot(data=t, x="Scénario", y="Résultat d'exploitation", color=viz.PRINCIPALE, ax=axes[0])
axes[0].axhline(resultat_depart, color=viz.GRIS, ls=":", lw=1.2)
axes[0].text(-0.45, resultat_depart, f"réel {EXERCICE}", va="bottom", ha="left", fontsize=9, color=viz.GRIS)
viz.format_dollars(axes[0]); axes[0].set_xlabel(""); axes[0].set_ylabel("")
viz.titre(axes[0], f"Résultat d'exploitation {ANNEE} par scénario", "Base : lissage saisonnier par canal")
sns.barplot(data=t, x="Scénario", y="Magasins déficitaires", color=viz.ALERTE, ax=axes[1])
axes[1].set_xlabel(""); axes[1].set_ylabel(""); axes[1].set_ylim(0, 54)
viz.titre(axes[1], "Magasins déficitaires", "Sur 54 magasins")
for ax in axes:
    ax.tick_params(axis="x", rotation=20)
fig.tight_layout()
viz.enregistrer(fig, "14_scenarios")
plt.show()

**Lecture.**
- **Le scénario prudent** limite la casse (+9,4 M$) sans changer la situation des magasins : 52 restent déficitaires.
- **Le scénario charges** ramène le résultat à son niveau de 2021 (140,0 M$) : il efface les hausses, mais pas la baisse des ventes.
- **Le scénario assortiment** rapporte deux fois plus (+53,2 M$) et dépasse 2021, mais 49 magasins restent déficitaires : les nouveautés se vendent surtout en ligne.
- **Le scénario complet** est le seul qui rende les magasins bénéficiaires dans leur ensemble (+8,0 M$) : 19 magasins restent déficitaires sur 54, contre 52 au fil de l'eau. Il rapporte 79,0 M$, un peu plus que la somme de ses leviers pris un à un (75,2 M$), parce que les nouveautés mises en avant en magasin sont plus nombreuses quand on en lance davantage.

C'est la traduction chiffrée de la recommandation de l'UA2 : **traiter la cause et la conséquence ensemble**.

## 6. La recommandation tient-elle si la prévision se trompe ?

C'est la question qui rend le simulateur utile malgré une prévision à ±10 %. On compare deux trajectoires sorties du même moteur : si la base est fausse, elle l'est dans les deux, et l'écart, qui est la valeur de la décision, reste juste.

### 6.1 Cinq bases différentes

In [ ]:
variantes = {f"{nom}" : (base, 0.0) for nom, base in BASES.items()}
variantes["Lissage saisonnier −10 %"] = (BASE, -0.10)
variantes["Lissage saisonnier +10 %"] = (BASE, +0.10)

rangs, gains, synth = {}, {}, []
for nom, (base, erreur) in variantes.items():
    g = gains_leviers(base, erreur_base=erreur)
    gains[nom] = g["Gain par an"]
    rangs[nom] = g["Gain par an"].rank(ascending=False).astype(int)
    s, _ = jouer(base, erreur_base=erreur)
    synth.append({"Base": nom, "Fil de l'eau : résultat": s.loc["Au fil de l'eau", "Résultat d'exploitation"],
                  "Fil de l'eau : magasins déficitaires": s.loc["Au fil de l'eau", "Magasins déficitaires"],
                  "Complet : résultat": s.loc["Complet", "Résultat d'exploitation"],
                  "Complet : gain": s.loc["Complet", "Gain par rapport au fil de l'eau"],
                  "Complet : magasins déficitaires": s.loc["Complet", "Magasins déficitaires"]})
rangs = pd.DataFrame(rangs).loc[leviers.index]
gains = pd.DataFrame(gains).loc[leviers.index]
synth = pd.DataFrame(synth).set_index("Base")
print("Rang de chaque levier selon la base (1 = le plus rentable) :")
display(rangs)
print("Classement identique pour toutes les bases :", bool((rangs.nunique(axis=1) == 1).all()))
print("Gain de chaque levier selon la base :")
display(gains.map(m_dollars))
print("Fil de l'eau et scénario complet selon la base :")
display(synth.style.format({c: m_dollars for c in synth.columns if "résultat" in c or "gain" in c}))
tableaux["6_rangs_selon_base"] = rangs.reset_index()
tableaux["6_gains_selon_base"] = gains.reset_index()
tableaux["6_scenarios_selon_base"] = synth.reset_index()

**Lecture.**
- **Le classement des leviers est identique avec les cinq bases.** Ce n'est pas un hasard, et il faut le dire : les cinq premiers leviers (assortiment, charges, publicité) se chiffrent en dollars indépendants du volume de ventes. Seuls les leviers proportionnels aux ventes (remises, nouveautés en magasin, retours) bougent avec la base, de quelques centaines de milliers de dollars, et ils sont en bas du classement. **Les décisions qui comptent ne dépendent pas de la prévision.**
- **Le gain du scénario complet est stable** : de 78,2 à 80,4 M$ selon la base.
- **Ce qui dépend de la prévision, c'est le point d'arrivée** : avec le scénario complet, aucun magasin n'est déficitaire si les ventes tiennent (2021 reconduit, Random Forest, base +10 %), 19 avec la base retenue, 48 si la base est 10 % trop optimiste.

**La phrase à présenter** : *même si notre prévision se trompe de 10 %, la décision reste la même ; ce qui change, c'est le nombre de magasins qui redeviennent rentables dès 2022.*

### 6.2 Les hypothèses incertaines du scénario complet

Chaque hypothèse que les données ne permettent pas de fixer est poussée vers sa valeur défavorable, puis favorable, les autres restant à leur valeur centrale. Plus la barre est longue, plus l'hypothèse pèse sur le résultat.

In [ ]:
COMPLET = SCENARIOS["Complet"]
ref_complet = simulateur.resultat_entreprise(simulateur.simuler(donnees, BASE, reglages(**COMPLET)))
INCERTITUDES = {  # hypothèse -> (valeur défavorable, valeur favorable), libellés
    "Erreur de la base (±10 %)": ({"erreur_base": -0.10}, {"erreur_base": 0.10}),
    "Chiffre d'affaires d'une nouveauté (50 % à 150 % de celui de 2021)": ({"efficacite_nouveaute": 0.5}, {"efficacite_nouveaute": 1.5}),
    "Coût d'une nouveauté (150 000 $ à 0 $)": ({"cout_par_lancement": 150_000}, {"cout_par_lancement": 0}),
    "Réaction des ventes à la publicité (−5 % à 0 %)": ({"reaction_ventes_publicite": -0.05}, {"reaction_ventes_publicite": 0.0}),
    "Nouveautés en magasin (50 % à 100 % de l'écart comblé)": ({"nouveautes_en_magasin": 0.5}, {"nouveautes_en_magasin": 1.0}),
}
lignes = []
for nom, (bas, haut) in INCERTITUDES.items():
    valeurs = [simulateur.resultat_entreprise(simulateur.simuler(donnees, BASE, reglages(**{**COMPLET, **h}))) for h in (bas, haut)]
    lignes.append({"Hypothèse": nom, "Défavorable": valeurs[0], "Favorable": valeurs[1], "Amplitude": abs(valeurs[1] - valeurs[0])})
tornade = pd.DataFrame(lignes).set_index("Hypothèse").sort_values("Amplitude")

fig, ax = plt.subplots(figsize=(11, 4.2))
for y, (nom, l) in enumerate(tornade.iterrows()):
    ax.barh(y, l.Défavorable - ref_complet, left=ref_complet, color=viz.ALERTE)
    ax.barh(y, l.Favorable - ref_complet, left=ref_complet, color=viz.PRINCIPALE)
ax.axvline(ref_complet, color=viz.FONCE, lw=1.2)
ax.axvline(RESULTAT_FIL, color=viz.GRIS, ls=":", lw=1.2)
ax.text(RESULTAT_FIL, len(tornade) - 0.4, " fil de l'eau", color=viz.GRIS, fontsize=9)
ax.set_yticks(range(len(tornade))); ax.set_yticklabels([l.replace("$", r"\$") for l in tornade.index], fontsize=9)
viz.format_dollars(ax, axe="x"); ax.grid(axis="x"); ax.grid(axis="y", visible=False)
viz.titre(ax, f"Scénario complet : ce que change chaque hypothèse incertaine", f"Résultat d'exploitation {ANNEE} (trait plein : hypothèses centrales, rouge : défavorable, vert : favorable)")
viz.enregistrer(fig, "14_tornade")
plt.show()
display(tornade.sort_values("Amplitude", ascending=False).style.format(m_dollars))
tableaux["6_tornade"] = tornade.reset_index()

**Lecture.**
- **L'erreur de la base pèse le plus** : de 155,7 à 230,6 M$. Elle déplace tous les scénarios ensemble, sans changer la décision (partie 6.1).
- **Viennent ensuite les deux inconnues du levier assortiment** : ce que vendra une nouveauté (168,1 à 218,1 M$) et ce qu'elle coûte à concevoir et lancer (jusqu'à −42 M$ à 150 000 $ par nouveauté). Ce sont les deux chiffres à obtenir des achats et du marketing **avant** de décider.
- La réaction des ventes à la publicité et l'efficacité de la mise en rayon pèsent peu.
- **Dans le cas le plus défavorable testé** (151,1 M$), le scénario complet reste au-dessus du fil de l'eau (114,1 M$) de 37 M$.

## 7. Jusqu'où chaque décision reste-t-elle gagnante ?

Plutôt qu'un optimum, qui serait trivial puisque chaque levier rapporte plus quand on le pousse, on cherche la **zone de décision sûre** : pour quelles valeurs des inconnues la décision rapporte encore.

In [ ]:
# ---------------------------------------------------------------------------
# 1. Lancements : gain net de 280 nouveautés de plus, selon leur coût et leur chiffre d'affaires
# ---------------------------------------------------------------------------
couts = np.arange(0, 250_001, 25_000)
efficacites = np.round(np.arange(0.4, 1.61, 0.2), 1)
ref_fil = simulateur.resultat_entreprise(simulateur.simuler(donnees, BASE, reglages()))
grille_lanc = pd.DataFrame([[simulateur.resultat_entreprise(simulateur.simuler(donnees, BASE, reglages(lancements=552, efficacite_nouveaute=e, cout_par_lancement=c))) - ref_fil
                             for c in couts] for e in efficacites],
                           index=[pct(e, 0) for e in efficacites], columns=[f"{c // 1000} k$" for c in couts])
contribution = donnees["ca_par_nouveaute"] * (boutiques.marge_nette.sum() - boutiques[f"{simulateur.COMPTE_VARIABLE} {EXERCICE}"].sum()) / boutiques.ca_net.sum()
print(f"Seuil : une nouveauté qui vend comme celles de {EXERCICE} reste rentable tant qu'elle coûte moins de "
      f"{contribution:,.0f} $ à concevoir et lancer".replace(",", " "))

# ---------------------------------------------------------------------------
# 2. Publicité des magasins : gain selon la baisse du budget et la réaction des ventes
# ---------------------------------------------------------------------------
baisses = [0.05, 0.10, 0.15, 0.20, 0.30]
reactions = [0.0, -0.01, -0.02, -0.03, -0.05, -0.08, -0.10]
grille_pub = pd.DataFrame([[simulateur.resultat_entreprise(simulateur.simuler(donnees, BASE, reglages(baisse_publicite_magasins=b, reaction_ventes_publicite=r))) - ref_fil
                            for b in baisses] for r in reactions],
                          index=[pct(r, 0, signe=True) if r else "0 %" for r in reactions], columns=[f"−{b:.0%}".replace("%", " %") for b in baisses])

fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
for ax, g, titre, xl, yl in [(axes[0], grille_lanc, "280 nouveautés de plus : gain net", "Coût de conception et de lancement d'une nouveauté",
                              f"CA d'une nouveauté (100 % = {EXERCICE})"),
                             (axes[1], grille_pub, "Publicité des magasins : gain", "Baisse du budget", "Réaction des ventes des magasins")]:
    sns.heatmap(g / 1e6, annot=True, fmt=".0f", cmap=sns.diverging_palette(15, 160, s=60, as_cmap=True), center=0, cbar=False, ax=ax,
                annot_kws={"size": 8})
    ax.set_xlabel(xl); ax.set_ylabel(yl)
    viz.titre(ax, titre, "En millions de dollars de résultat par an · vert : décision gagnante, rouge : perdante")
fig.tight_layout()
viz.enregistrer(fig, "14_zones_de_decision")
plt.show()
tableaux["7_grille_lancements"] = grille_lanc.reset_index(names="CA d'une nouveauté")
tableaux["7_grille_publicite"] = grille_pub.reset_index(names="Réaction des ventes")

**Lecture.**
- **280 nouveautés de plus** restent gagnantes tant qu'une nouveauté coûte moins de **164 466 $** à concevoir et lancer, si elle vend comme celles de 2021. Si elle vend 40 % de moins, le seuil tombe vers 100 000 $ ; si elle vend 20 % de plus, il passe vers 200 000 $. Le chiffre à demander aux achats est donc simple : le coût complet d'un lancement.
- **Baisser la publicité des magasins de 10 %** reste gagnant tant que les ventes des magasins ne reculent pas de plus de 5 % ; à −20 % de budget, jusqu'à environ 10 % de ventes perdues. Aucun test n'a mesuré cette réaction (06) : la baisse doit être pilotée sur quelques magasins avant d'être généralisée.

## 8. Enregistrement

| Sortie | Contenu | Utilisée par |
|---|---|---|
| `data/outputs/scenarios/scenarios_2022.parquet` | Chaque scénario, boutique par boutique : CA, marge, charges, résultat, apport de chaque levier | Application Streamlit, rapport |
| `data/outputs/scenarios/base_canal_2022.parquet` | Base mensuelle 2022 par canal | Application Streamlit |
| `reports/livrables/L5_rapport_final/simulateur_tableaux.xlsx` | Tous les tableaux du notebook | Rapport final |

In [ ]:
# Un fichier par scénario et par boutique : ce que lira l'application Streamlit
colonnes = ["id_boutique", "nom_boutique", "type", "pays", "ca_net", "resultat_exploitation", "ca_net_simule",
            "marge_nette_simulee", "frais_generaux_simules", "publicite_simulee", "resultat_exploitation_simule"]
sortie = pd.concat([details[nom][colonnes + [c for c in details[nom].columns if c.startswith("apport : ")]].assign(scenario=nom, annee=ANNEE)
                    for nom in SCENARIOS], ignore_index=True)
sortie = sortie.rename(columns={"ca_net": f"ca_net_{EXERCICE}", "resultat_exploitation": f"resultat_exploitation_{EXERCICE}"})
config.SCENARIOS.mkdir(parents=True, exist_ok=True)
chemin_sortie = config.SCENARIOS / f"scenarios_{ANNEE}.parquet"
sortie.to_parquet(chemin_sortie, index=False)
lissage.to_parquet(config.SCENARIOS / f"base_canal_{ANNEE}.parquet")
chemin_excel = analyse.exporter_tableaux(tableaux, config.L5 / "simulateur_tableaux.xlsx")
for chemin in [chemin_sortie, config.SCENARIOS / f"base_canal_{ANNEE}.parquet", chemin_excel]:
    print("Enregistré :", chemin.relative_to(config.ROOT))

## 9. Synthèse

### 1. Le coût de l'inaction
Au fil de l'eau, le résultat d'exploitation tombe de **141,7 M$ à 114,1 M$** en 2022, et **52 magasins sur 54** deviennent déficitaires. La moitié de la perte vient de la baisse des ventes, l'autre des hausses de charges reconduites.

### 2. Les décisions, classées
| Rang | Levier | Gain 2022 |
|---|---|---|
| 1 | Revenir au rythme de lancement de 2019 | 46,1 M$ (gain brut) |
| 2 | Revenir aux charges de 2019 (personnel, administration, informatique) | 15,1 M$ |
| 3 | Regrouper les bureaux par pays (piste, non cumulable avec le rang 2) | 7,5 M$ |
| 4 | Baisser la publicité des magasins de 10 % | 6,2 M$ |
| 5 | Geler les mêmes charges au niveau de 2021 | 6,1 M$ |
| 6 à 9 | Remises en magasin, nouveautés en rayon, plafond de remise hors magasin, retours | 2,5 à 4,6 M$ chacun |

**Le scénario complet** (assortiment et charges) rapporte **79,0 M$** et rend les magasins bénéficiaires dans leur ensemble : 19 déficitaires au lieu de 52.

### 3. Pourquoi on peut s'y fier malgré une prévision à ±10 %
- Le moteur refait l'exercice 2021 et les onze leviers de l'analyse au dollar près.
- Le classement des leviers est identique avec cinq bases différentes, et le gain du scénario complet varie de 78,2 à 80,4 M$.
- Même avec l'hypothèse la plus défavorable testée, le scénario complet reste 37 M$ au-dessus du fil de l'eau.
- Ce qui reste incertain, c'est le point d'arrivée des magasins (de 0 à 48 déficitaires selon la base), pas la décision.

### 4. Ce qu'il faut obtenir avant de décider
1. **Le coût complet d'un lancement** : sous 164 466 $, renouveler l'assortiment est gagnant.
2. **Un test de baisse de publicité** sur quelques magasins : au-delà de 5 % de ventes perdues, la baisse de 10 % ne paie plus.

### 5. Pour la suite
- **Notebook 13** : quelles sous-catégories renouveler en priorité, et avec quel budget d'achat, pour atteindre les 552 nouveautés.
- **Notebook 10** : sa conclusion peut maintenant s'écrire ainsi : la prévision sert de base de comparaison aux scénarios, à ±10 %, et cette marge ne change pas la décision.
- **Application Streamlit** : les réglages de la partie 4 deviennent des curseurs, sur le même moteur.